# HumanProof: train the voice and face models on a GPU

Runtime → Change runtime type → **GPU** (A100 or L4 on Colab Pro).

Put the datasets in Google Drive first (each requires accepting its licence):

| Model | Dataset | Where to request it |
|---|---|---|
| Voice anti-spoof | ASVspoof 2019 LA (+ PA optional) | datashare.ed.ac.uk (ASVspoof 2019) |
| Voice anti-spoof, real-world test | In-the-Wild | deepfake-total.com/in_the_wild |
| Face deepfake | FaceForensics++ (c23) | github.com/ondyari/FaceForensics (request form) |
| Face deepfake, cross-dataset test | Celeb-DF v2 | github.com/yuezunli/celeb-deepfakeforensics (request form) |

Most of these licences are **research / non-commercial**. Use the resulting models for research and demos; retrain on data you have commercial rights to before selling the product.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/humanproof-data'   # edit to your folder
OUT  = '/content/drive/MyDrive/humanproof-models'

In [ ]:
# Private repository? Use https://<token>@github.com/Minnu27/humanproof.git
!git clone https://github.com/Minnu27/humanproof.git /content/repo
%cd /content/repo/humanproof
!pip -q install -r ml/requirements-gpu.txt

## 0. Add the sessions your own app has collected (optional, recommended)
Pulls the labelled tester sessions out of your database and decrypts them onto this Colab disk (not Google Drive). Paste the same `HP_DATABASE_URL` and `HP_MASTER_SECRET` the deployment uses; they are kept in memory only. Skip this section to train on the public datasets alone.

The export is unencrypted biometric data: the last cell of this notebook deletes it.

In [ ]:
import os, getpass
os.environ['HP_DATABASE_URL'] = getpass.getpass('HP_DATABASE_URL: ')
os.environ['HP_MASTER_SECRET'] = getpass.getpass('HP_MASTER_SECRET: ')
!pip -q install "psycopg[binary,pool]" pydantic-settings cryptography
!python ml/dataset/export.py --out /content/hp-export
del os.environ['HP_DATABASE_URL'], os.environ['HP_MASTER_SECRET']
COLLECTED = os.path.exists('/content/hp-export/voice.csv')
VOICE_EXTRA = '--csv /content/hp-export/voice.csv' if COLLECTED else ''
FACE_EXTRA = '--extra /content/hp-export/faces' if COLLECTED else ''

In [ ]:
# How the live checks did on those sessions, and whether there is enough to fine-tune on
!python ml/retrain.py --data /content/hp-export --dry-run

## 1. Voice anti-spoofing (≈3–6 h on an A100 for 30 epochs)

In [ ]:
!python ml/voice/train_antispoof.py --asvspoof2019-la $DATA/LA --in-the-wild $DATA/release_in_the_wild --epochs 30 --batch 32 $VOICE_EXTRA --out $OUT

## 2. Speaker embedding (pretrained ECAPA-TDNN, exported to ONNX; minutes)

In [ ]:
!python ml/voice/export_speaker.py --out $OUT

## 3. Face deepfake detector
First extract face crops the same way the app does (MediaPipe Face Landmarker).

In [ ]:
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!python ml/face/extract_faces.py --ffpp $DATA/FaceForensics++ --celebdf $DATA/Celeb-DF-v2 --landmarker face_landmarker.task --out /content/faces --frames 16 --workers 8

In [ ]:
!python ml/face/train_face.py --faces /content/faces --epochs 12 --batch 64 $FACE_EXTRA --out $OUT --cross-test celebdf

## 4. Check the reports, then copy the models to the server
`$OUT` now holds `*.onnx`, `*.metrics.json` and `manifest.json` (with SHA-256 hashes). Merge `manifest.json` entries with the motor model's and copy everything into `backend/models/` on the server. The backend refuses any model whose hash does not match the manifest.

In [ ]:
import json, glob
for f in sorted(glob.glob(OUT + '/*.metrics.json')):
    print(f, json.dumps(json.load(open(f)), indent=2))

## 5. Delete the decrypted dataset

In [ ]:
!rm -rf /content/hp-export /content/faces && echo deleted